# 01 — Core / heavy analysis

**Public reproducibility version.** This notebook preserves the final analytical logic used for the revised manuscript. Local absolute paths and development-only cells were cleaned for portability. The notebook has not been re-executed as part of repository packaging.


# 01 — Core / Heavy Analysis (NO_BRAIN_FOG)

**Purpose:** create the frozen revised analysis base after removing only three broad,
non-ADRD-specific inclusion triggers: `brain fog`, `foggy brain`, and `cognitive fog`.

This notebook performs the computationally expensive steps:

- refined 17-keyword cohort construction,
- interaction expansion,
- social-network construction and community detection,
- centrality/network metrics,
- text preprocessing and deduplication,
- SBERT embeddings,
- semantic graph + Leiden clustering,
- corrected semantic-node → processed-text → expanded-row topic propagation,
- semantic mapping integrity checks,
- Top-5 semantic-topic content audit,
- threshold sensitivity analysis from 0.60 to 0.85.

**Boundary:** user-topic/bipartite, no-topic characterization, interaction/community
comparisons, cross-network chi-square analyses, effect sizes, and final manuscript
figures are handled in `02_downstream_analysis_NO_BRAIN_FOG_FINAL.ipynb`.


> **Corrected descriptive-output version:** the analytical pipeline and section
> structure are unchanged. This version fixes only the final keyword-frequency
> denominator/matching consistency and adds explicit original-post interaction
> summaries for manuscript revision.


## ⚙️ Configuration — Edit paths here

In [ ]:
import networkx, sentence_transformers, scipy, statsmodels, pandas, numpy, matplotlib, seaborn
import sys, igraph, leidenalg

print("Python:", sys.version)
for pkg in [networkx, sentence_transformers, scipy, statsmodels, pandas, numpy, matplotlib, seaborn, igraph]:
    print(pkg.__name__, pkg.__version__)

# Report leidenalg separately
print("leidenalg", leidenalg.version)

In [ ]:
import os
from pathlib import Path

# Repository-relative paths. Run the notebook from notebooks/ or the repository root.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

DATA_DIR       = str(PROJECT_ROOT / "data" / "processed")
FOLLOWERS_PATH = str(PROJECT_ROOT / "data" / "raw" / "followers.csv")
NLTK_DIR       = str(PROJECT_ROOT / "data" / "nltk_data")
REVISION_ROOT  = str(PROJECT_ROOT)

RUN_NAME = "main_pipeline_no_brain_fog_final_v1"
SAVE_DIR = os.path.join(REVISION_ROOT, "outputs", RUN_NAME)
CACHE_DIR = SAVE_DIR
PARQUET_FILE = "matched_subset_exact_fuzzy_no_political_5_7_new.parquet"

SEMANTIC_THRESHOLD = 0.70
LEIDEN_SEED         = 42
BIPARTITE_K         = 5
BIPARTITE_FRAC      = 0.5

FORCE_RECOMPUTE = True
RUN_HEAVY_SNA_METRICS = True
RUN_THRESHOLD_SENSITIVITY = True
REBUILD_SEMANTIC_MAPPING = True

# Keep the historical fresh-run safety guard. The output folder must be empty.
STRICT_FRESH_RUN = True

if os.path.isdir(SAVE_DIR):
    existing = [name for name in os.listdir(SAVE_DIR) if not name.startswith(".")]
else:
    existing = []

if STRICT_FRESH_RUN and existing:
    raise RuntimeError(
        "Fresh-run output directory is not empty:\n"
        f"{SAVE_DIR}\n\n"
        "Delete/rename the output folder or change RUN_NAME before a new clean run."
    )

os.makedirs(SAVE_DIR, exist_ok=True)
raw_path = os.path.join(DATA_DIR, PARQUET_FILE)

print("Input parquet       :", raw_path)
print("Followers file      :", FOLLOWERS_PATH)
print("Revision output dir :", SAVE_DIR)


### Fresh-run rule

Run this notebook with **Restart Kernel and Run All Cells**.

The folder

`04_revision_outputs/main_pipeline_no_brain_fog_final_v1`

must be empty before the run. If a previous attempt exists, rename or delete that
folder first. The strict guard prevents old and new heavy-analysis outputs from being mixed.

After a successful run, treat this folder as **frozen upstream output** and run Notebook 02.


## Mapping fixes in this version

1. **Semantic mapping:** semantic communities are estimated on unique `text_processed` values, then the topic is explicitly propagated back to every expanded row sharing that processed text.
2. **Integrity audit:** verifies one topic per processed text, one topic per original `post_id` after interaction expansion, and correct `interaction_type` ↔ `partner_id` alignment.
3. **Follower relation:** calculated against the current expanded row's `partner_id` only.
4. **Community relation:** calculated against the current expanded row's `partner_id` only; other reply/repost/quote targets from the original unsplit post are not mixed into that row.
5. **Topic × community relation:** topics are joined by explicit `original_index`; `topic=-1` rows are excluded from topic-community cross-tabulation.

The weighted CNM social-network construction itself is **not changed** in this version.


## Refined keyword specification

This run differs from the mapping-fixed fresh run in **one cohort-definition change only**:

- removed: `brain fog`
- removed: `foggy brain`
- removed: `cognitive fog`

A post containing one of these phrases is **still retained** if it also matches any of the remaining 17 ADRD keywords. Thus this notebook removes the three phrases as independent inclusion triggers; it does not automatically exclude every post in which those phrases appear.


---
## Part 1 — Import & Filter Data

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import re

# Load raw data
raw_path = os.path.join(DATA_DIR, PARQUET_FILE)
filtered_posts_df = pq.read_table(raw_path).to_pandas()
filtered_posts_df["text"] = filtered_posts_df["text"].astype(str)

print(f"Raw rows: {filtered_posts_df.shape[0]:,}")
filtered_posts_df.info()

### 1.1 — Keyword Filtering

In [ ]:
# ADRD keyword list — refined specification
#
# ONLY THREE inclusion triggers are removed relative to the original 20-keyword run:
#   "brain fog", "foggy brain", "cognitive fog"
#
# IMPORTANT:
# We re-filter the SAME source parquet by raw post text.
# Because the refined keyword set is a subset of the original set, the existing
# parquet contains all posts that can qualify under this refined specification.

ORIGINAL_KEYWORDS_20 = [
    "Alzheimer", "dementia", "ADRD",
    "brain fog", "cognitive decline", "forgetfulness", "cognitive test", "memory loss",
    "apoe", "amyloid", "Aducanumab", "Donanemab", "Lecanemab",
    "Donepezil", "Memantine", "Risperidone", "foggy brain",
    "cognitive fog", "mild cognitive", "cognitive training"
]

REMOVED_NONSPECIFIC_KEYWORDS = [
    "brain fog",
    "foggy brain",
    "cognitive fog",
]

keywords = [
    kw for kw in ORIGINAL_KEYWORDS_20
    if kw.lower() not in {x.lower() for x in REMOVED_NONSPECIFIC_KEYWORDS}
]

exact_match_keywords = {"ADRD", "apoe"}
exact_match_lower = {k.lower() for k in exact_match_keywords}

def contains_AD_keywords_text(text, keyword_list, exact_lower):
    text = "" if pd.isna(text) else str(text)
    return any(
        re.search(
            rf"\b{re.escape(kw)}\b" if kw.lower() in exact_lower
            else rf"{re.escape(kw)}[\u2019's]*",
            text,
            flags=re.IGNORECASE
        )
        for kw in keyword_list
    )

# Old 20-keyword eligibility, used only to quantify how many rows are removed.
old_keyword_mask = filtered_posts_df["text"].apply(
    lambda x: contains_AD_keywords_text(x, ORIGINAL_KEYWORDS_20, exact_match_lower)
)

# Refined 17-keyword eligibility — THIS defines the revised cohort.
refined_keyword_mask = filtered_posts_df["text"].apply(
    lambda x: contains_AD_keywords_text(x, keywords, exact_match_lower)
)

filtered_posts_df["AD_related"] = refined_keyword_mask

# Keep only ADRD-related posts; preserve the original political-noise exclusion.
filtered_AD_related = filtered_posts_df[filtered_posts_df["AD_related"]].copy()
filtered_AD_related = filtered_AD_related[
    ~filtered_AD_related["text"].str.contains(
        "mcconnell|reagan|ebony", case=False, na=False
    )
].copy()

# Audit the keyword-specification change BEFORE political-noise exclusion.
n_old = int(old_keyword_mask.sum())
n_refined = int(refined_keyword_mask.sum())
n_removed_by_keyword_change = int((old_keyword_mask & ~refined_keyword_mask).sum())
n_contains_removed_but_still_retained = int(
    (
        filtered_posts_df["text"].str.contains(
            "|".join(re.escape(x) for x in REMOVED_NONSPECIFIC_KEYWORDS),
            case=False, na=False, regex=True
        )
        & refined_keyword_mask
    ).sum()
)

print("=== Refined ADRD Keyword Specification ===")
print("Original keyword count :", len(ORIGINAL_KEYWORDS_20))
print("Refined keyword count  :", len(keywords))
print("Removed triggers       :", REMOVED_NONSPECIFIC_KEYWORDS)
print()
print(f"Rows matching old 20-keyword definition     : {n_old:,}")
print(f"Rows matching refined 17-keyword definition : {n_refined:,}")
print(f"Rows removed by keyword refinement          : {n_removed_by_keyword_change:,}")
print(
    "Rows containing a removed phrase but retained because another ADRD keyword also matched : "
    f"{n_contains_removed_but_still_retained:,}"
)

# FINAL keyword-frequency summary for the 17 retained keywords.
#
# IMPORTANT:
# Numerator and denominator are both calculated in the FINAL retained cohort
# AFTER the political-noise exclusion. The same keyword-matching helper used
# to define cohort eligibility is reused here so the descriptive frequencies
# are internally consistent.
match_counts = {}
for kw in keywords:
    match_counts[kw] = filtered_AD_related["text"].apply(
        lambda t: contains_AD_keywords_text(
            t,
            [kw],
            exact_match_lower
        )
    ).sum()

df_keywords = pd.DataFrame(
    list(match_counts.items()),
    columns=["Keyword", "Matched_Row_Count"]
)

total_rows = filtered_AD_related.shape[0]
df_keywords["Proportion"] = (
    df_keywords["Matched_Row_Count"] / total_rows
    if total_rows else np.nan
)
df_keywords = df_keywords.sort_values(
    "Proportion",
    ascending=False
).reset_index(drop=True)

from IPython.display import display
display(df_keywords)

top_kw = df_keywords.iloc[0]
print(f"\n✅ FINAL ADRD posts retained after political exclusion: {total_rows:,}")
print(
    f"🔝 Top retained keyword: '{top_kw['Keyword']}' — "
    f"{int(top_kw['Matched_Row_Count']):,} rows ({top_kw['Proportion']:.2%})"
)
print(
    "ℹ️ Keyword percentages are not mutually exclusive because a post can match "
    "more than one retained keyword."
)

# Save the exact descriptive keyword-frequency table used for Results.
df_keywords.to_csv(
    os.path.join(
        SAVE_DIR,
        "final_17_keyword_frequency_after_political_exclusion.csv"
    ),
    index=False,
)

# Save a small audit table for the revision record.
keyword_refinement_audit = pd.DataFrame({
    "metric": [
        "original_keyword_count",
        "refined_keyword_count",
        "rows_matching_old_20_keyword_definition",
        "rows_matching_refined_17_keyword_definition",
        "rows_removed_by_keyword_refinement",
        "rows_containing_removed_phrase_but_retained_via_other_keyword",
        "rows_retained_after_political_exclusion",
    ],
    "value": [
        len(ORIGINAL_KEYWORDS_20),
        len(keywords),
        n_old,
        n_refined,
        n_removed_by_keyword_change,
        n_contains_removed_but_still_retained,
        int(filtered_AD_related.shape[0]),
    ],
})
keyword_refinement_audit.to_csv(
    os.path.join(SAVE_DIR, "keyword_refinement_no_brain_fog_audit.csv"),
    index=False,
)


### 1.2 — Expand Interaction Rows (one row per interaction type)

In [ ]:
from collections import Counter

# Count interaction types BEFORE splitting
def detect_interaction_types(row):
    return [
        label
        for col, label in zip(
            ["replied_author", "reposted_author", "quoted_author"],
            ["reply",          "repost",          "quote"]
        )
        if pd.notnull(row[col])
    ]

before_counts = Counter(
    i for lst in filtered_AD_related.apply(detect_interaction_types, axis=1) for i in lst
)
before_df = pd.DataFrame(before_counts.most_common(), columns=["edge_type", "count"])
before_df["proportion"] = before_df["count"] / before_df["count"].sum()
print("🔎 Interaction types BEFORE splitting:")
print(before_df)

# Original-post level interaction summary BEFORE row expansion.
has_interaction = filtered_AD_related.apply(
    lambda row: any(pd.notnull(row[c]) for c in [
        "replied_author", "reposted_author", "quoted_author"
    ]),
    axis=1
)

n_original_posts = len(filtered_AD_related)
n_interaction_posts = int(has_interaction.sum())
n_noninteraction_posts = int((~has_interaction).sum())

interaction_post_summary = pd.DataFrame([
    {
        "category": "interaction_post",
        "count": n_interaction_posts,
        "proportion": n_interaction_posts / n_original_posts
    },
    {
        "category": "non_interaction_post",
        "count": n_noninteraction_posts,
        "proportion": n_noninteraction_posts / n_original_posts
    }
])

print("\n📌 Original-post interaction summary BEFORE splitting:")
print(f"Total original posts   : {n_original_posts:,}")
print(
    f"Interaction posts      : {n_interaction_posts:,} "
    f"({n_interaction_posts / n_original_posts:.2%})"
)
print(
    f"Non-interaction posts  : {n_noninteraction_posts:,} "
    f"({n_noninteraction_posts / n_original_posts:.2%})"
)

interaction_post_summary.to_csv(
    os.path.join(SAVE_DIR, "original_post_interaction_summary.csv"),
    index=False
)
before_df.to_csv(
    os.path.join(SAVE_DIR, "interaction_type_instances_before_splitting.csv"),
    index=False
)

# Expand: one row per interaction type
def split_interactions(row):
    rows = []
    for col, label in zip(
        ["replied_author", "reposted_author", "quoted_author"],
        ["reply",          "repost",          "quote"]
    ):
        if pd.notnull(row[col]):
            new = row.copy()
            new["partner_id"]       = row[col]
            new["interaction_type"] = label
            rows.append(new)
    return rows if rows else [row]

expanded = filtered_AD_related.apply(split_interactions, axis=1)
filtered_AD_related = pd.DataFrame([r for lst in expanded for r in lst]).reset_index(drop=True)

after_counts = (
    filtered_AD_related["interaction_type"]
    .value_counts()
    .reset_index()
    .rename(columns={"interaction_type": "edge_type", "count": "count"})
)
after_counts["proportion"] = after_counts["count"] / after_counts["count"].sum()
print("\n📈 Interaction types AFTER splitting:")
print(after_counts)
print(f"\nTotal rows after expansion: {filtered_AD_related.shape[0]:,}")

after_counts.to_csv(
    os.path.join(SAVE_DIR, "interaction_type_instances_after_splitting.csv"),
    index=False
)

### 1.3 — Follower Network Enrichment

In [ ]:
# Collect all user IDs involved
uids       = set(filtered_AD_related["user_id"])
reply_ids  = set(filtered_AD_related["replied_author"].dropna().astype(int))
repost_ids = set(filtered_AD_related["reposted_author"].dropna().astype(int))
quote_ids  = set(filtered_AD_related["quoted_author"].dropna().astype(int))
all_ids    = uids | reply_ids | repost_ids | quote_ids

# Load followers and filter to relevant users
followers_df = pd.read_csv(FOLLOWERS_PATH, names=["follower", "followed"])
filtered_followers_df = followers_df[
    followers_df["follower"].isin(all_ids) & followers_df["followed"].isin(all_ids)
].reset_index(drop=True)
follower_set = set(zip(filtered_followers_df["follower"], filtered_followers_df["followed"]))

print(f"Follower pairs retained: {len(follower_set):,}")

In [ ]:
def determine_follower_relation(row, fset):
    """
    Row-level follower relationship AFTER interaction expansion.

    0 = no follow relationship / no interaction partner
    1 = user follows current partner
    2 = current partner follows user
    3 = mutual followers

    IMPORTANT:
    Each expanded row represents one reply/repost/quote interaction.
    Therefore this function uses only `partner_id` for that row.
    It must NOT inspect replied_author/reposted_author/quoted_author together,
    because those columns still retain the original post-level values and can
    contain multiple partners from the unsplit post.
    """
    uid = row["user_id"]
    partner = row.get("partner_id")

    if pd.isnull(partner):
        return 0

    uid = int(uid)
    partner = int(partner)

    fwd = (uid, partner) in fset
    rev = (partner, uid) in fset

    if fwd and rev:
        return 3
    if fwd:
        return 1
    if rev:
        return 2
    return 0


filtered_AD_related["follower_direction_type"] = filtered_AD_related.apply(
    lambda row: determine_follower_relation(row, follower_set), axis=1
)

print("Follower-direction type counts (row-level current partner only):")
print(filtered_AD_related["follower_direction_type"].value_counts().sort_index())


In [ ]:
# Merge follower / following degree counts onto main dataframe
follower_counts = followers_df.groupby("followed").size().rename("num_followers")
following_counts = followers_df.groupby("follower").size().rename("num_followings")
user_degree_df = (
    pd.concat([follower_counts, following_counts], axis=1)
    .fillna(0).astype(int)
    .reset_index()
    .rename(columns={"index": "user_id"})
)

# Partner degree
partner_deg = user_degree_df.rename(columns={
    "user_id": "partner_user_id",
    "num_followers": "partner_num_followers",
    "num_followings": "partner_num_followings"
})
filtered_AD_related = filtered_AD_related.merge(
    partner_deg, how="left", left_on="partner_id", right_on="partner_user_id"
).drop(columns=["partner_user_id"])

# User degree
user_deg = user_degree_df.rename(columns={
    "num_followers": "user_num_followers",
    "num_followings": "user_num_followings"
})
filtered_AD_related = filtered_AD_related.merge(user_deg, how="left", on="user_id")

# Missing value summary
missing = filtered_AD_related.isnull().sum()
missing_pct = (missing / len(filtered_AD_related) * 100).round(2)
print("🔍 Missing Value Summary:")
print(pd.DataFrame({"Missing_Count": missing, "Missing_%": missing_pct})
      .query("Missing_Count > 0")
      .sort_values("Missing_Count", ascending=False))

---
## Part 2 — Social Network Analysis (People-to-People)

In [ ]:
import networkx as nx

# Build directed weighted graph
edge_df = (
    filtered_AD_related[["user_id", "partner_id", "interaction_type"]]
    .dropna()
    .copy()
    .rename(columns={"user_id": "source", "partner_id": "target", "interaction_type": "edge_type"})
)
edge_df["source"] = edge_df["source"].astype(int)
edge_df["target"] = edge_df["target"].astype(int)

edge_df_weighted = (
    edge_df.groupby(["source", "target", "edge_type"]).size().reset_index(name="weight")
)

G = nx.DiGraph()
for _, row in edge_df_weighted.iterrows():
    G.add_edge(row["source"], row["target"],
               weight=row["weight"], edge_type=row["edge_type"])

print(f"Nodes: {G.number_of_nodes():,}   Edges: {G.number_of_edges():,}")

# Edge type proportions
edge_counts = edge_df["edge_type"].value_counts().reset_index()
edge_counts.columns = ["edge_type", "count"]
edge_counts["proportion"] = edge_counts["count"] / edge_counts["count"].sum()
print(edge_counts)

In [ ]:
# Connected component sizes
G_undirected = G.to_undirected()
cc_sorted = sorted(nx.connected_components(G_undirected), key=len, reverse=True)
cc_df = pd.DataFrame([
    {"component_id": i, "size": len(c)} for i, c in enumerate(cc_sorted[:10])
])
print(cc_df)
print(f"Total components: {nx.number_connected_components(G_undirected):,}")

In [ ]:
# Extract Largest Connected Component (LCC) — directed
lcc_nodes = max(nx.weakly_connected_components(G), key=len)
G_lcc     = G.subgraph(lcc_nodes).copy()

total_weight_lcc = sum(d["weight"] for _, _, d in G_lcc.edges(data=True))
print(f"LCC nodes : {G_lcc.number_of_nodes():,}  ({G_lcc.number_of_nodes()/G.number_of_nodes():.2%})")
print(f"LCC edges : {G_lcc.number_of_edges():,}  ({G_lcc.number_of_edges()/G.number_of_edges():.2%})")
print(f"Total edge weight in LCC: {total_weight_lcc:,}")

### 2.1 — Community Detection (CNM / FastGreedy)

In [ ]:
import os, igraph as ig

_cnm_rev = os.path.join(SAVE_DIR, "user_community_df.csv")
_mod_rev = os.path.join(SAVE_DIR, "modularity_cnm.txt")
_cnm_old = os.path.join(CACHE_DIR, "user_community_df.csv")
_mod_old = os.path.join(CACHE_DIR, "modularity_cnm.txt")

if (not FORCE_RECOMPUTE) and os.path.exists(_cnm_rev) and os.path.exists(_mod_rev):
    _cnm_read, _mod_read = _cnm_rev, _mod_rev
elif (not FORCE_RECOMPUTE) and os.path.exists(_cnm_old) and os.path.exists(_mod_old):
    _cnm_read, _mod_read = _cnm_old, _mod_old
else:
    _cnm_read = _mod_read = None

if _cnm_read is not None:
    user_community_df = pd.read_csv(_cnm_read)
    with open(_mod_read) as _f:
        modularity_cnm = float(_f.read().strip())
    community_sizes = user_community_df["community"].value_counts().reset_index()
    community_sizes.columns = ["community", "num_nodes"]
    print("CACHE: CNM loaded from:", _cnm_read)
    print(f"Modularity={modularity_cnm:.4f}  Communities={community_sizes.shape[0]}")
else:
    ig_edges = [(u, v, d.get("weight", 1)) for u, v, d in G_lcc.to_undirected().edges(data=True)]
    ig_graph = ig.Graph.TupleList(ig_edges, weights=True, directed=False)
    cnm = ig_graph.community_fastgreedy(weights=ig_graph.es["weight"]).as_clustering()
    ig_graph.vs["community"] = cnm.membership
    ig_graph.vs["label"] = ig_graph.vs["name"]
    modularity_cnm = cnm.modularity

    user_community_df = pd.DataFrame({
        "user_id": ig_graph.vs["name"],
        "community": cnm.membership
    })
    community_sizes = user_community_df["community"].value_counts().reset_index()
    community_sizes.columns = ["community", "num_nodes"]

    total_n = user_community_df.shape[0]
    top20 = community_sizes.head(20)["num_nodes"].sum()

    print(f"Modularity (CNM): {modularity_cnm:.4f}")
    print(f"Communities: {community_sizes.shape[0]}  Top20: {top20} ({top20/total_n:.2%})")

    ig_graph.write_gml(os.path.join(SAVE_DIR, "community_network.gml"))
    user_community_df.to_csv(_cnm_rev, index=False)
    with open(_mod_rev, "w") as _f:
        _f.write(str(modularity_cnm))
    print("SAVED revised CNM outputs to:", SAVE_DIR)


In [ ]:
# Intra-community edge type breakdown
edge_with_comm = edge_df_weighted.merge(
    user_community_df, left_on="source", right_on="user_id", how="left"
).merge(
    user_community_df, left_on="target", right_on="user_id", how="left",
    suffixes=("_src", "_tgt")
)

internal_edges = edge_with_comm[
    edge_with_comm["community_src"] == edge_with_comm["community_tgt"]
].copy()
internal_edges["community"] = internal_edges["community_src"]

# Per-community ratios
edge_type_pivot = (
    internal_edges.groupby(["community", "edge_type"])["weight"]
    .sum()
    .reset_index()
    .pivot(index="community", columns="edge_type", values="weight")
    .fillna(0)
)
edge_type_pivot["total"] = edge_type_pivot.sum(axis=1)
for t in ["reply", "quote", "repost"]:
    if t not in edge_type_pivot.columns:
        edge_type_pivot[t] = 0
    edge_type_pivot[f"{t}_ratio"] = edge_type_pivot[t] / edge_type_pivot["total"]

def get_dominant_type(row):
    return max(["reply", "quote", "repost"], key=lambda t: row.get(f"{t}_ratio", 0))

edge_type_pivot["dominant_type"] = edge_type_pivot.apply(get_dominant_type, axis=1)
edge_type_pivot = edge_type_pivot.reset_index().merge(community_sizes, on="community", how="left")

all_communities_with_type = edge_type_pivot[[
    "community", "num_nodes", "reply_ratio", "quote_ratio", "repost_ratio", "dominant_type"
]].copy()
print("Top 20 communities — intra edge types:")
print(all_communities_with_type.sort_values("num_nodes", ascending=False).head(20))

# Dominant type distribution
dominant_counts = all_communities_with_type["dominant_type"].value_counts().reset_index()
dominant_counts.columns = ["dominant_type", "count"]
dominant_counts["proportion"] = dominant_counts["count"] / dominant_counts["count"].sum()
print("\nDominant type distribution:")
print(dominant_counts)

In [ ]:
# Inter-community edge type breakdown
inter_edges = edge_with_comm[
    edge_with_comm["community_src"] != edge_with_comm["community_tgt"]
].copy()

inter_type = (
    inter_edges.groupby("edge_type")["weight"].sum().reset_index()
)
inter_type["proportion"] = inter_type["weight"] / inter_type["weight"].sum()
print("🔗 Inter-community edge type ratios:")
print(inter_type)

# Intra-community overall
intra_type = (
    internal_edges.groupby("edge_type")["weight"].sum().reset_index()
)
intra_type["proportion"] = intra_type["weight"] / intra_type["weight"].sum()
print("\n🏠 Intra-community edge type ratios:")
print(intra_type)

### 2.2 — Network Metrics (LCC)

In [ ]:
import os
from networkx.linalg.algebraicconnectivity import algebraic_connectivity

_cen_rev = os.path.join(SAVE_DIR, "centrality_df.csv")
_met_rev = os.path.join(SAVE_DIR, "network_metrics.csv")
_cen_old = os.path.join(CACHE_DIR, "centrality_df.csv")
_met_old = os.path.join(CACHE_DIR, "network_metrics.csv")

if (not FORCE_RECOMPUTE) and os.path.exists(_cen_rev) and os.path.exists(_met_rev):
    _cen_read, _met_read = _cen_rev, _met_rev
elif (not FORCE_RECOMPUTE) and os.path.exists(_cen_old) and os.path.exists(_met_old):
    _cen_read, _met_read = _cen_old, _met_old
else:
    _cen_read = _met_read = None

if _cen_read is not None:
    centrality_df = pd.read_csv(_cen_read)
    network_metrics = pd.read_csv(_met_read, index_col=0)
    print("CACHE: Centrality loaded from:", _cen_read)
    print(centrality_df.drop(columns=["user_id"]).describe().T.round(4))
else:
    degree_dict      = dict(G_lcc.degree())
    in_degree_dict   = dict(G_lcc.in_degree())
    out_degree_dict  = dict(G_lcc.out_degree())
    betweenness_dict = nx.betweenness_centrality(G_lcc, weight="weight")
    closeness_dict   = nx.closeness_centrality(G_lcc)
    eigenvector_dict = nx.eigenvector_centrality(G_lcc, max_iter=1000)

    ig_edges2 = [(u, v, d.get("weight", 1)) for u, v, d in G_lcc.to_undirected().edges(data=True)]
    ig_graph2 = ig.Graph.TupleList(ig_edges2, weights=True, directed=False)
    ig_graph2.vs["name"] = [str(v) for v in ig_graph2.vs["name"]]
    comm_map2 = dict(zip(user_community_df["user_id"].astype(str), user_community_df["community"]))
    ig_graph2.vs["community"] = [comm_map2.get(v["name"], -1) for v in ig_graph2.vs]
    constraint_list = ig_graph2.constraint()

    centrality_df = pd.DataFrame({
        "user_id"     : list(G_lcc.nodes()),
        "Degree"      : [degree_dict.get(n, 0)       for n in G_lcc.nodes()],
        "In-degree"   : [in_degree_dict.get(n, 0)    for n in G_lcc.nodes()],
        "Out-degree"  : [out_degree_dict.get(n, 0)   for n in G_lcc.nodes()],
        "Betweenness" : [betweenness_dict.get(n, 0.) for n in G_lcc.nodes()],
        "Closeness"   : [closeness_dict.get(n, 0.)   for n in G_lcc.nodes()],
        "Eigenvector" : [eigenvector_dict.get(n, 0.) for n in G_lcc.nodes()],
        "Constraint"  : constraint_list
    })

    # Save centrality only to the revision folder.
    centrality_df.to_csv(_cen_rev, index=False)
    print("SAVED revised centrality to:", _cen_rev)


In [ ]:
# Network-level metrics.
# Reuse the saved metrics by default because recomputing shortest paths and
# algebraic connectivity is not needed for the no-topic audit.
if (not RUN_HEAVY_SNA_METRICS) and ("network_metrics" in globals()) and network_metrics is not None:
    print("NETWORK-LEVEL STRUCTURE METRICS (cached)")
    print("=" * 50)
    print(network_metrics)
else:
    G_lcc_und     = G_lcc.to_undirected()
    density       = nx.density(G_lcc)
    avg_clust     = nx.average_clustering(G_lcc_und)
    transitivity  = nx.transitivity(G_lcc_und)
    assortativity = nx.degree_assortativity_coefficient(G_lcc)
    reciprocity   = nx.reciprocity(G_lcc)

    num_sl   = len(list(nx.selfloop_edges(G_lcc)))
    sl_ratio = num_sl / G_lcc.number_of_edges()

    deg_vals    = centrality_df["Degree"]
    max_deg     = deg_vals.max()
    n_nodes     = len(deg_vals)
    deg_central = sum(max_deg - deg_vals) / ((n_nodes - 1) * (n_nodes - 2)) if n_nodes > 2 else None

    if nx.is_connected(G_lcc_und):
        avg_path = nx.average_shortest_path_length(G_lcc_und)
        diameter = nx.diameter(G_lcc_und)
        radius   = nx.radius(G_lcc_und)
    else:
        avg_path = diameter = radius = None

    alg_conn = algebraic_connectivity(G_lcc_und)

    network_metrics = pd.DataFrame.from_dict({
        "Number of Nodes"              : G_lcc.number_of_nodes(),
        "Number of Edges"              : G_lcc.number_of_edges(),
        "Density"                      : density,
        "Average Clustering Coeff"     : avg_clust,
        "Transitivity"                 : transitivity,
        "Avg Shortest Path Length"     : avg_path,
        "Diameter"                     : diameter,
        "Radius"                       : radius,
        "Degree Centralization"        : deg_central,
        "Self-Loop Count"              : num_sl,
        "Self-Loop Proportion"         : round(sl_ratio, 4),
        "Assortativity Coefficient"    : assortativity,
        "Reciprocity"                  : reciprocity,
        "Modularity (CNM)"             : modularity_cnm,
        "Algebraic Connectivity"       : alg_conn,
    }, orient="index", columns=["Value"])

    print("NETWORK-LEVEL STRUCTURE METRICS (recomputed)")
    print("=" * 50)
    print(network_metrics)
    network_metrics.to_csv(os.path.join(SAVE_DIR, "network_metrics.csv"))


---
## Part 3 — Text Preprocessing

In [ ]:
import nltk
import numpy as np
import matplotlib.pyplot as plt
from nltk.tokenize import TreebankWordTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# NLTK setup — downloads once to NLTK_DIR
os.environ["NLTK_DATA"] = NLTK_DIR
nltk.data.path.insert(0, NLTK_DIR)
for pkg in ["punkt", "wordnet", "omw-1.4", "stopwords"]:
    nltk.download(pkg, download_dir=NLTK_DIR, quiet=True)

_tokenizer  = TreebankWordTokenizer()
_lemmatizer = WordNetLemmatizer()
_stopwords  = set(stopwords.words("english"))

def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)   # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)             # keep letters only
    tokens = _tokenizer.tokenize(text)
    tokens = [t for t in tokens if t not in _stopwords]
    return " ".join(_lemmatizer.lemmatize(t) for t in tokens)

# Work on English-only posts
filtered_posts_df = filtered_AD_related.copy()
filtered_posts_df = filtered_posts_df[
    filtered_posts_df["langs"].apply(lambda x: "eng" in x)
].copy()

# Parallel preprocess (pandarallel) — falls back to .apply if unavailable
try:
    from pandarallel import pandarallel
    pandarallel.initialize(progress_bar=True, verbose=0)
    filtered_posts_df["text_processed"] = filtered_posts_df["text"].parallel_apply(preprocess)
except Exception:
    print("pandarallel not available — using standard apply (slower)")
    filtered_posts_df["text_processed"] = filtered_posts_df["text"].apply(preprocess)

# Binary URL / hashtag indicators
filtered_posts_df["has_url"]     = filtered_posts_df["text"].apply(
    lambda x: int(bool(re.search(r"http\S+|www\.\S+", str(x))))
)
filtered_posts_df["has_hashtag"] = filtered_posts_df["text"].apply(
    lambda x: int(bool(re.search(r"#\w+", str(x))))
)

print(f"Posts after English filter: {filtered_posts_df.shape[0]:,}")
print(filtered_posts_df["interaction_type"].value_counts())

In [ ]:
# Deduplicate for semantic analysis
filtered_posts_df = filtered_posts_df.reset_index(drop=True)
filtered_posts_df["original_index"] = filtered_posts_df.index

nonempty_mask = (
    filtered_posts_df["text_processed"].notna() &
    (filtered_posts_df["text_processed"].str.strip() != "")
)

filtered_posts_df_nonempty = filtered_posts_df[nonempty_mask].copy()
dedup_df = filtered_posts_df_nonempty.drop_duplicates(
    subset=["text_processed"]
).copy()

docs_all = dedup_df["text_processed"].astype(str).str.strip().tolist()

# Word length
doc_lengths = [len(d.split()) for d in docs_all]

# Character length
char_lengths = [len(d) for d in docs_all]

print(f"✅ Unique non-empty posts: {len(docs_all):,}")
print(f"📌 Original rows: {len(filtered_posts_df):,}")

print(f"Avg length: {np.mean(doc_lengths):.1f} words")
print(
    f"Q1={np.percentile(doc_lengths,25):.0f}  "
    f"Q3={np.percentile(doc_lengths,75):.0f}  "
    f"IQR={np.percentile(doc_lengths,75)-np.percentile(doc_lengths,25):.0f}"
)

print(f"🔤 Max character length: {max(char_lengths):,} characters")

# Optional: show the longest processed post
max_char_idx = np.argmax(char_lengths)
print("\nLongest processed post:")
print(docs_all[max_char_idx])

plt.hist(doc_lengths, bins=30, edgecolor="black")
plt.title("Distribution of Post Lengths (words)")
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

---
## Part 4 — Semantic Analysis (SBERT + Leiden)

In [ ]:
import os, numpy as np
from sklearn.preprocessing import normalize

_emb_rev = os.path.join(SAVE_DIR, "embeddings_revision.npy")
_emb_old = os.path.join(CACHE_DIR, "embeddings.npy")

embeddings = None
if not FORCE_RECOMPUTE:
    for candidate in [_emb_rev, _emb_old]:
        if os.path.exists(candidate):
            _tmp = np.load(candidate)
            if _tmp.shape[0] == len(docs_all):
                embeddings = _tmp
                print("CACHE: Embeddings loaded from:", candidate)
                break
            else:
                print(
                    "Ignoring embedding cache with incompatible row count:",
                    candidate, _tmp.shape, "expected", len(docs_all)
                )

if embeddings is None:
    from sentence_transformers import SentenceTransformer
    sbert_model = SentenceTransformer("all-mpnet-base-v2")
    embeddings = sbert_model.encode(
        docs_all, batch_size=64, show_progress_bar=True
    )
    embeddings = normalize(embeddings, norm="l2")
    np.save(_emb_rev, embeddings)
    print("SAVED revised embeddings:", _emb_rev)

# Normalize defensively because cosine thresholding assumes unit-length vectors.
embeddings = normalize(np.asarray(embeddings), norm="l2").astype(np.float32)
print("Embedding shape:", embeddings.shape)


In [ ]:
import os
import numpy as np
import pandas as pd
from collections import Counter

_map_path = os.path.join(SAVE_DIR, "semantic_node_topic_map_revision.csv")
_mod_path = os.path.join(SAVE_DIR, "leiden_modularity_revision.txt")
_edge_path = os.path.join(SAVE_DIR, "semantic_edge_count_revision.txt")

_use_revised_map = (
    (not FORCE_RECOMPUTE)
    and (not REBUILD_SEMANTIC_MAPPING)
    and os.path.exists(_map_path)
    and os.path.exists(_mod_path)
)

if _use_revised_map:
    semantic_node_topic_map = pd.read_csv(_map_path)
    nodes_in_graph = semantic_node_topic_map["semantic_node_id"].astype(int).tolist()
    labels = semantic_node_topic_map["topic"].astype(int).tolist()
    modularity_sem = float(open(_mod_path).read().strip())

    community_sizes_sem = Counter(labels)
    top5_communities = community_sizes_sem.most_common(5)

    print("CACHE: Explicit revised semantic mapping loaded.")
    print("Nodes={:,}  Communities={}  Modularity={:.4f}".format(
        len(nodes_in_graph), len(community_sizes_sem), modularity_sem
    ))
else:
    import igraph as ig
    import leidenalg

    # Build cosine-threshold edges in blocks to avoid allocating a full 9,985 x 9,985 matrix.
    # Embeddings are L2-normalized, so dot product equals cosine similarity.
    block_size = 512
    n_docs = embeddings.shape[0]
    edges_sem = []

    print(
        "Rebuilding semantic graph with explicit node IDs "
        f"(threshold={SEMANTIC_THRESHOLD:.2f})..."
    )

    for start in range(0, n_docs, block_size):
        stop = min(start + block_size, n_docs)
        sims = embeddings[start:stop] @ embeddings.T

        for local_i in range(stop - start):
            i = start + local_i
            if i + 1 >= n_docs:
                continue
            row = sims[local_i]
            js = np.flatnonzero(row[i + 1:] >= SEMANTIC_THRESHOLD) + (i + 1)
            edges_sem.extend(
                (int(i), int(j), float(row[j]))
                for j in js
            )

        print(f"  processed {stop:,}/{n_docs:,} documents", end="\r")

    print()
    if len(edges_sem) == 0:
        raise RuntimeError("No semantic edges were found at the selected threshold.")

    G_ig = ig.Graph.TupleList(
        edges_sem,
        edge_attrs=["weight"],
        directed=False,
        vertex_name_attr="name"
    )

    partition = leidenalg.find_partition(
        G_ig,
        leidenalg.ModularityVertexPartition,
        weights="weight",
        seed=LEIDEN_SEED
    )

    # CRITICAL FIX:
    # partition.membership follows igraph's internal vertex order.
    # Map each membership label to the actual semantic node ID stored in vertex 'name'.
    vertex_node_ids = np.asarray([int(v) for v in G_ig.vs["name"]], dtype=int)
    raw_labels = np.asarray(partition.membership, dtype=int)

    # Canonicalize topic IDs by descending community size so IDs are stable and interpretable.
    raw_sizes = Counter(raw_labels.tolist())
    raw_order = [cid for cid, _ in raw_sizes.most_common()]
    canonical = {raw_cid: rank for rank, raw_cid in enumerate(raw_order)}
    canonical_labels = np.asarray([canonical[x] for x in raw_labels], dtype=int)

    node_to_topic = dict(zip(vertex_node_ids.tolist(), canonical_labels.tolist()))
    nodes_in_graph = sorted(node_to_topic)
    labels = [node_to_topic[node_id] for node_id in nodes_in_graph]

    modularity_sem = float(partition.modularity)
    community_sizes_sem = Counter(labels)
    top5_communities = community_sizes_sem.most_common(5)

    # Map semantic-node position -> representative original row explicitly.
    dedup_original_indices = dedup_df["original_index"].to_numpy()
    if max(nodes_in_graph) >= len(dedup_original_indices):
        raise IndexError("Semantic node ID exceeds deduplicated dataframe bounds.")

    semantic_node_topic_map = pd.DataFrame({
        "semantic_node_id": nodes_in_graph,
        "original_index": dedup_original_indices[np.asarray(nodes_in_graph, dtype=int)],
        "topic": labels,
    })

    semantic_node_topic_map.to_csv(_map_path, index=False)
    with open(_mod_path, "w") as f:
        f.write(str(modularity_sem))
    with open(_edge_path, "w") as f:
        f.write(str(len(edges_sem)))

    print("Semantic graph rebuilt.")
    print("Nodes      : {:,}".format(len(nodes_in_graph)))
    print("Edges      : {:,}".format(len(edges_sem)))
    print("Communities:", len(community_sizes_sem))
    print("Modularity : {:.4f}".format(modularity_sem))
    print("Explicit mapping saved:", _map_path)

print("Top 5 semantic communities:")
for rank, (cid, size) in enumerate(top5_communities, 1):
    print("  {}. Topic {}: {} ({:.2%})".format(
        rank, cid, size, size / len(labels)
    ))


In [ ]:
import os, numpy as np
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# Representative posts for each top-5 community
index_map       = {i: dedup_df["original_index"].tolist()[i] for i in range(len(dedup_df))}
original_texts  = filtered_posts_df["text"].astype(str).tolist()

print("📌 Top 10 representative posts per community:\n")
for rank, (cid, _) in enumerate(top5_communities, 1):
    print(f"── Community {cid} (rank {rank}) ──")
    node_ids         = [nid for nid, lbl in zip(nodes_in_graph, labels) if lbl == cid]
    cluster_embs     = embeddings[node_ids]
    centroid         = cluster_embs.mean(axis=0, keepdims=True)
    sims             = cosine_similarity(cluster_embs, centroid).flatten()
    top_local        = np.argsort(sims)[-10:][::-1]
    for j, li in enumerate(top_local):
        orig_idx = index_map[node_ids[li]]
        print(f"  {j+1}. {original_texts[orig_idx].strip()}")
    print()

---
## Top-5 Semantic Topic Content Audit

The centroid-nearest posts shown above are useful for naming a topic, but they can
make a community look cleaner than it really is. This audit therefore samples
each of the five largest semantic communities in three ways:

1. **Centroid-nearest posts** — highly representative of the community center.
2. **Random posts** — a less selected view of ordinary community members.
3. **Boundary posts** — posts with lower similarity to the community centroid,
   useful for checking whether the narrative remains coherent at the edges.

The output is intended for manual interpretation of topic clarity. Topic IDs are
reported without imposing the old manuscript narrative labels.


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

TOPIC_AUDIT_RANDOM_SEED = 20260826
N_CENTROID = 10
N_RANDOM = 15
N_BOUNDARY = 5

rng = np.random.default_rng(TOPIC_AUDIT_RANDOM_SEED)

# semantic_node_topic_map is the corrected semantic-node -> original-row mapping.
# dedup_df contains one representative row per unique preprocessed text.
dedup_text_lookup = (
    dedup_df[
        ["original_index", "text_processed"]
    ]
    .drop_duplicates(subset=["original_index"])
    .copy()
)

original_text_lookup = (
    filtered_posts_df[
        ["original_index", "text"]
    ]
    .drop_duplicates(subset=["original_index"])
    .copy()
)

topic_posts = (
    semantic_node_topic_map[
        ["semantic_node_id", "original_index", "topic"]
    ]
    .merge(dedup_text_lookup, on="original_index", how="left", validate="one_to_one")
    .merge(original_text_lookup, on="original_index", how="left", validate="one_to_one")
)

audit_rows = []

for rank, (topic_id, topic_size) in enumerate(top5_communities, start=1):
    grp = topic_posts.loc[topic_posts["topic"] == topic_id].copy()
    node_ids = grp["semantic_node_id"].astype(int).to_numpy()

    if len(node_ids) == 0:
        continue

    cluster_embs = embeddings[node_ids]
    centroid = cluster_embs.mean(axis=0, keepdims=True)
    centroid_sims = cosine_similarity(cluster_embs, centroid).ravel()

    grp["centroid_similarity"] = centroid_sims
    grp = grp.sort_values("centroid_similarity", ascending=False).reset_index(drop=True)

    # 1) Centroid-nearest.
    centroid_take = grp.head(min(N_CENTROID, len(grp))).copy()
    centroid_take["sample_type"] = "centroid_nearest"

    # 2) Boundary / low-centroid-similarity posts.
    remaining_after_centroid = grp.loc[
        ~grp["semantic_node_id"].isin(centroid_take["semantic_node_id"])
    ].copy()

    boundary_take = (
        remaining_after_centroid
        .sort_values("centroid_similarity", ascending=True)
        .head(min(N_BOUNDARY, len(remaining_after_centroid)))
        .copy()
    )
    boundary_take["sample_type"] = "boundary_low_centroid_similarity"

    # 3) Random ordinary members, excluding already selected posts.
    remaining = remaining_after_centroid.loc[
        ~remaining_after_centroid["semantic_node_id"].isin(boundary_take["semantic_node_id"])
    ].copy()

    random_n = min(N_RANDOM, len(remaining))
    if random_n > 0:
        random_indices = rng.choice(remaining.index.to_numpy(), size=random_n, replace=False)
        random_take = remaining.loc[random_indices].copy()
    else:
        random_take = remaining.head(0).copy()

    random_take["sample_type"] = "random_member"

    sampled = pd.concat(
        [centroid_take, random_take, boundary_take],
        ignore_index=True
    )

    sampled["topic_rank"] = rank
    sampled["topic_size"] = int(topic_size)
    sampled["within_topic_sample_order"] = np.arange(1, len(sampled) + 1)

    audit_rows.append(sampled)

top5_topic_content_audit = pd.concat(audit_rows, ignore_index=True)

# De-identify the review file: do not include user_id or partner_id.
top5_topic_content_audit = top5_topic_content_audit[
    [
        "topic_rank",
        "topic",
        "topic_size",
        "sample_type",
        "within_topic_sample_order",
        "centroid_similarity",
        "text",
        "text_processed",
        "semantic_node_id",
        "original_index",
    ]
].sort_values(
    ["topic_rank", "sample_type", "within_topic_sample_order"]
)

audit_csv = os.path.join(
    SAVE_DIR,
    "top5_semantic_topic_content_audit_30_each.csv"
)
top5_topic_content_audit.to_csv(audit_csv, index=False)

# Also create a reviewer-friendly Excel workbook with one sheet per topic.
audit_xlsx = os.path.join(
    SAVE_DIR,
    "top5_semantic_topic_content_audit_30_each.xlsx"
)

with pd.ExcelWriter(audit_xlsx, engine="openpyxl") as writer:
    summary = (
        top5_topic_content_audit
        .groupby(["topic_rank", "topic", "topic_size", "sample_type"])
        .size()
        .reset_index(name="n_sampled")
    )
    summary.to_excel(writer, sheet_name="summary", index=False)

    for rank, topic_id in (
        top5_topic_content_audit[["topic_rank", "topic"]]
        .drop_duplicates()
        .sort_values("topic_rank")
        .itertuples(index=False, name=None)
    ):
        sheet_df = top5_topic_content_audit.loc[
            top5_topic_content_audit["topic"] == topic_id
        ].copy()
        sheet_df.to_excel(
            writer,
            sheet_name=f"topic_{int(topic_id)}",
            index=False
        )

print("=== Top-5 Semantic Topic Content Audit ===")
print(
    top5_topic_content_audit
    .groupby(["topic_rank", "topic", "topic_size", "sample_type"])
    .size()
    .reset_index(name="n_sampled")
    .to_string(index=False)
)

print("\nSaved:")
print(" ", audit_csv)
print(" ", audit_xlsx)

# Display the random and boundary posts because they are most informative for topic clarity.
for rank, topic_id in (
    top5_topic_content_audit[["topic_rank", "topic"]]
    .drop_duplicates()
    .sort_values("topic_rank")
    .itertuples(index=False, name=None)
):
    print(f"\n--- Topic {int(topic_id)} (rank {int(rank)}) ---")
    display_cols = ["sample_type", "centroid_similarity", "text"]
    display(
        top5_topic_content_audit.loc[
            (top5_topic_content_audit["topic"] == topic_id)
            & (top5_topic_content_audit["sample_type"] != "centroid_nearest"),
            display_cols
        ].reset_index(drop=True)
    )


---
## Part 5 — User–Topic Bipartite Network

In [ ]:
import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# A. Reconstruct the LEGACY mapping for audit only.
# ---------------------------------------------------------------------------
# The legacy notebook treated semantic node IDs (positions in dedup_df)
# as if they were original row indices in the expanded dataframe.
# This is retained ONLY to quantify the old mapping error.
legacy_post_topic_df = pd.DataFrame({
    "original_index": np.asarray(nodes_in_graph, dtype=int),
    "topic": np.asarray(labels, dtype=int),
})

legacy_filtered_posts_df_2 = filtered_posts_df.merge(
    legacy_post_topic_df,
    on="original_index",
    how="left",
    validate="one_to_one",
    sort=False,
)
legacy_filtered_posts_df_2["topic"] = (
    pd.to_numeric(legacy_filtered_posts_df_2["topic"], errors="coerce")
    .fillna(-1)
    .astype(int)
)

# ---------------------------------------------------------------------------
# B. Correct semantic-node -> representative processed-text mapping.
# ---------------------------------------------------------------------------
semantic_node_topic_map = semantic_node_topic_map.copy()

# `semantic_node_topic_map.original_index` points to the representative row
# in dedup_df for each semantic node.
representative_topic_map = semantic_node_topic_map.merge(
    filtered_posts_df_nonempty[["original_index", "text_processed"]],
    on="original_index",
    how="left",
    validate="one_to_one",
    sort=False,
)

if representative_topic_map["text_processed"].isna().any():
    raise RuntimeError(
        "Mapping failure: at least one semantic node has no representative text_processed."
    )

# Because semantic clustering was performed after deduplicating by text_processed,
# each semantic node must correspond to exactly one unique processed text.
if representative_topic_map["text_processed"].duplicated().any():
    dup = representative_topic_map.loc[
        representative_topic_map["text_processed"].duplicated(keep=False),
        ["semantic_node_id", "original_index", "text_processed", "topic"],
    ]
    dup.to_csv(
        os.path.join(SAVE_DIR, "ERROR_duplicate_representative_semantic_texts.csv"),
        index=False,
    )
    raise RuntimeError(
        "Mapping failure: representative semantic texts are not unique after deduplication."
    )

text_topic_map = (
    representative_topic_map[["text_processed", "topic"]]
    .drop_duplicates()
    .copy()
)

# One processed text must never map to multiple semantic topics.
topic_counts_per_text = (
    text_topic_map.groupby("text_processed")["topic"].nunique()
)
if (topic_counts_per_text > 1).any():
    bad_texts = topic_counts_per_text[topic_counts_per_text > 1].index
    text_topic_map.loc[
        text_topic_map["text_processed"].isin(bad_texts)
    ].to_csv(
        os.path.join(SAVE_DIR, "ERROR_processed_text_multiple_topics.csv"),
        index=False,
    )
    raise RuntimeError(
        "Mapping failure: at least one text_processed maps to multiple topics."
    )

# ---------------------------------------------------------------------------
# C. Propagate semantic topics to ALL expanded rows sharing text_processed.
# ---------------------------------------------------------------------------
# This is intentional:
#   * semantic clustering counts each unique processed text once;
#   * reply/repost/quote expansion can create several rows for the same post/text;
#   * all those expanded rows should inherit the same semantic topic.
#
# Use Series.map rather than a dataframe merge so row order and original_index
# are preserved exactly.
topic_by_text = text_topic_map.set_index("text_processed")["topic"]

filtered_posts_df_2 = filtered_posts_df.copy()
filtered_posts_df_2["topic"] = (
    filtered_posts_df_2["text_processed"]
    .map(topic_by_text)
    .fillna(-1)
    .astype(int)
)

# ---------------------------------------------------------------------------
# D. Hard integrity checks for semantic mapping.
# ---------------------------------------------------------------------------
if len(filtered_posts_df_2) != len(filtered_posts_df):
    raise RuntimeError("Row-count changed during topic propagation.")

if not filtered_posts_df_2["original_index"].is_unique:
    raise RuntimeError("original_index is not unique in the expanded analytic dataframe.")

if not np.array_equal(
    filtered_posts_df_2["original_index"].to_numpy(),
    filtered_posts_df["original_index"].to_numpy(),
):
    raise RuntimeError("Row order/original_index changed during topic propagation.")

# Same processed text must always receive one and only one topic value.
nonempty_audit = filtered_posts_df_2.loc[
    filtered_posts_df_2["text_processed"].notna()
    & filtered_posts_df_2["text_processed"].astype(str).str.strip().ne("")
].copy()

same_text_topic_n = (
    nonempty_audit.groupby("text_processed")["topic"].nunique()
)
if (same_text_topic_n > 1).any():
    raise RuntimeError(
        "Integrity failure: identical text_processed values received different topics."
    )

# Same original Bluesky post_id may appear multiple times after reply/repost/quote
# expansion, but all expanded rows from that post must receive the same topic.
post_topic_n = filtered_posts_df_2.groupby("post_id")["topic"].nunique()
if (post_topic_n > 1).any():
    bad_post_ids = post_topic_n[post_topic_n > 1].index
    filtered_posts_df_2.loc[
        filtered_posts_df_2["post_id"].isin(bad_post_ids),
        [
            "post_id", "original_index", "interaction_type", "partner_id",
            "text", "text_processed", "topic"
        ],
    ].to_csv(
        os.path.join(SAVE_DIR, "ERROR_same_post_multiple_topics.csv"),
        index=False,
    )
    raise RuntimeError(
        "Integrity failure: expanded rows from the same post_id received different topics."
    )

# ---------------------------------------------------------------------------
# E. Hard audit of reply/repost/quote expansion.
# ---------------------------------------------------------------------------
# The current row's partner_id must agree with the interaction-specific source
# column that created that expanded row.
interaction_partner_column = {
    "reply": "replied_author",
    "repost": "reposted_author",
    "quote": "quoted_author",
}

partner_alignment_bad = []

for interaction_type, source_col in interaction_partner_column.items():
    mask = filtered_posts_df_2["interaction_type"].eq(interaction_type)
    chk = filtered_posts_df_2.loc[
        mask,
        ["post_id", "original_index", "interaction_type", "partner_id", source_col]
    ].copy()

    left = pd.to_numeric(chk["partner_id"], errors="coerce")
    right = pd.to_numeric(chk[source_col], errors="coerce")

    bad = ~(left.eq(right) | (left.isna() & right.isna()))
    if bad.any():
        bad_df = chk.loc[bad].copy()
        bad_df["expected_partner_column"] = source_col
        partner_alignment_bad.append(bad_df)

if partner_alignment_bad:
    partner_alignment_bad = pd.concat(partner_alignment_bad, ignore_index=True)
    partner_alignment_bad.to_csv(
        os.path.join(SAVE_DIR, "ERROR_interaction_partner_alignment.csv"),
        index=False,
    )
    raise RuntimeError(
        "Interaction expansion failure: partner_id does not match the row's interaction type."
    )

# ---------------------------------------------------------------------------
# F. Diagnostics describing duplicate-text propagation.
# ---------------------------------------------------------------------------
processed_text_multiplicity = (
    nonempty_audit
    .groupby("text_processed")
    .agg(
        n_expanded_rows=("original_index", "size"),
        n_post_ids=("post_id", "nunique"),
        n_authors=("user_id", "nunique"),
        n_topics=("topic", "nunique"),
    )
    .reset_index()
)

# These are not errors. They document cases where identical processed text
# corresponds to several expanded interaction rows and/or several original posts.
multirow_texts = processed_text_multiplicity.query("n_expanded_rows > 1").copy()
multipost_texts = processed_text_multiplicity.query("n_post_ids > 1").copy()

mapping_compare = pd.DataFrame({
    "semantic_node_id": np.asarray(nodes_in_graph, dtype=int),
    "legacy_original_index": np.asarray(nodes_in_graph, dtype=int),
    "correct_original_index": semantic_node_topic_map["original_index"].to_numpy(dtype=int),
    "topic": np.asarray(labels, dtype=int),
})
mapping_compare["legacy_row_equals_correct_row"] = (
    mapping_compare["legacy_original_index"]
    == mapping_compare["correct_original_index"]
)

legacy_topic_rows = int((legacy_filtered_posts_df_2["topic"] >= 0).sum())
correct_topic_rows = int((filtered_posts_df_2["topic"] >= 0).sum())

mapping_summary = pd.DataFrame(
    {
        "metric": [
            "expanded_rows_total",
            "semantic_nodes_with_topics",
            "legacy_topic_assigned_rows",
            "correct_topic_assigned_rows",
            "unique_nonempty_processed_texts",
            "processed_texts_represented_by_multiple_expanded_rows",
            "processed_texts_shared_by_multiple_post_ids",
            "post_ids_with_multiple_expanded_rows",
            "post_ids_with_multiple_topic_values",
            "interaction_partner_alignment_errors",
        ],
        "value": [
            len(filtered_posts_df_2),
            len(nodes_in_graph),
            legacy_topic_rows,
            correct_topic_rows,
            nonempty_audit["text_processed"].nunique(),
            len(multirow_texts),
            len(multipost_texts),
            int((filtered_posts_df_2.groupby("post_id").size() > 1).sum()),
            int((post_topic_n > 1).sum()),
            0,
        ],
    }
)

print("=== Corrected Semantic Mapping Integrity Audit ===")
print(mapping_summary.to_string(index=False))
print()
print(
    "Legacy semantic-node ID matched the correct original row in "
    "{:,}/{:,} cases ({:.2%}).".format(
        mapping_compare["legacy_row_equals_correct_row"].sum(),
        len(mapping_compare),
        mapping_compare["legacy_row_equals_correct_row"].mean(),
    )
)
print(
    "\nPASS: identical processed texts have one topic; "
    "expanded rows from the same post_id have one topic; "
    "interaction_type and partner_id are aligned."
)

# ---------------------------------------------------------------------------
# G. Save audit artifacts to the fresh revision output folder.
# ---------------------------------------------------------------------------
mapping_compare.to_csv(
    os.path.join(SAVE_DIR, "legacy_vs_correct_semantic_row_mapping.csv"),
    index=False,
)
filtered_posts_df_2.to_parquet(
    os.path.join(SAVE_DIR, "filtered_posts_df_2_corrected_propagated.parquet"),
    index=False,
)
text_topic_map.to_csv(
    os.path.join(SAVE_DIR, "text_to_topic_map_corrected.csv"),
    index=False,
)
mapping_summary.to_csv(
    os.path.join(SAVE_DIR, "semantic_mapping_integrity_summary.csv"),
    index=False,
)
multirow_texts.to_csv(
    os.path.join(SAVE_DIR, "processed_texts_with_multiple_expanded_rows.csv"),
    index=False,
)
multipost_texts.to_csv(
    os.path.join(SAVE_DIR, "processed_texts_shared_across_post_ids.csv"),
    index=False,
)

print("\nCorrected mapping and integrity-audit outputs saved to:", SAVE_DIR)


---
# Final Core Step — Threshold Sensitivity Analysis

**Goal:** recompute the semantic graph/community solution at cosine-similarity
thresholds 0.60–0.85 and save the sensitivity table used by Notebook 02.

This is intentionally kept in Notebook 01 because it depends on the frozen SBERT
embeddings and requires repeated semantic-graph/Leiden computation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

THRESHOLDS = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85]
PAPER_THRESHOLD = 0.70

_sens_old = os.path.join(CACHE_DIR, "threshold_sensitivity_summary.csv")
_sens_rev = os.path.join(SAVE_DIR, "threshold_sensitivity_summary.csv")

if (not RUN_THRESHOLD_SENSITIVITY) and os.path.exists(_sens_old):
    summary_df = pd.read_csv(_sens_old)
    results = summary_df.to_dict("records")
    print("Threshold sensitivity loaded from original cache:", _sens_old)
elif (not RUN_THRESHOLD_SENSITIVITY) and os.path.exists(_sens_rev):
    summary_df = pd.read_csv(_sens_rev)
    results = summary_df.to_dict("records")
    print("Threshold sensitivity loaded from revision cache:", _sens_rev)
else:
    import igraph as ig
    import leidenalg
    from collections import Counter

    def build_edges_at_threshold(thr, block_size=512):
        edges = []
        n_docs = embeddings.shape[0]
        for start in range(0, n_docs, block_size):
            stop = min(start + block_size, n_docs)
            sims = embeddings[start:stop] @ embeddings.T
            for local_i in range(stop - start):
                i = start + local_i
                if i + 1 >= n_docs:
                    continue
                row = sims[local_i]
                js = np.flatnonzero(row[i + 1:] >= thr) + (i + 1)
                edges.extend(
                    (int(i), int(j), float(row[j]))
                    for j in js
                )
        return edges

    def run_threshold(thr, seed=42):
        edges = build_edges_at_threshold(thr)
        n_total = embeddings.shape[0]
        n_in_graph = len({v for e in edges for v in e[:2]})
        if not edges:
            return {
                "threshold": thr,
                "node_retention": 0,
                "n_nodes": 0,
                "n_edges": 0,
                "n_communities": 0,
                "modularity": np.nan,
                "top1_share": np.nan,
                "top5_share": np.nan,
            }

        graph = ig.Graph.TupleList(
            edges,
            edge_attrs=["weight"],
            directed=False,
            vertex_name_attr="name"
        )
        part = leidenalg.find_partition(
            graph,
            leidenalg.ModularityVertexPartition,
            weights="weight",
            seed=seed
        )
        labels_t = part.membership
        comm_sizes = Counter(labels_t)
        top5 = comm_sizes.most_common(5)

        return {
            "threshold": thr,
            "node_retention": round(n_in_graph / n_total, 4),
            "n_nodes": n_in_graph,
            "n_edges": len(edges),
            "n_communities": len(comm_sizes),
            "modularity": round(part.modularity, 4),
            "top1_share": round(top5[0][1] / len(labels_t), 4) if top5 else np.nan,
            "top5_share": round(sum(s for _, s in top5) / len(labels_t), 4) if top5 else np.nan,
        }

    results = []
    for thr in THRESHOLDS:
        print(f"threshold={thr} ...", end=" ", flush=True)
        result = run_threshold(thr, seed=LEIDEN_SEED)
        results.append(result)
        print(
            f"nodes={result['n_nodes']:,}  "
            f"Q={result['modularity']}  "
            f"top5={result['top5_share']:.1%}"
        )

    summary_df = pd.DataFrame(results)
    summary_df.to_csv(_sens_rev, index=False)
    print("Recomputed threshold sensitivity saved to:", _sens_rev)


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(results)

display_df = summary_df.copy()
display_df["node_retention"] = display_df["node_retention"].map("{:.1%}".format)
display_df["top1_share"]     = display_df["top1_share"].map("{:.1%}".format)
display_df["top5_share"]     = display_df["top5_share"].map("{:.1%}".format)

def highlight_row(row):
    bg = "background-color: #d4edda" if row["threshold"] == PAPER_THRESHOLD else ""
    return [bg] * len(row)

display(
    display_df.style
    .apply(highlight_row, axis=1)
    .set_caption(f"Threshold Sensitivity — green row = selected threshold ({PAPER_THRESHOLD})")
)

summary_df.to_csv(f"{SAVE_DIR}/threshold_sensitivity_summary.csv", index=False)
print("✅ Table saved.")

In [ ]:
# ── Dual-axis plot: Modularity Q + Node Retention ────────────────────────────
thr_vals = summary_df["threshold"].tolist()
q_vals   = summary_df["modularity"].tolist()
ret_vals = (summary_df["node_retention"] * 100).tolist()  # as %
paper_idx = thr_vals.index(PAPER_THRESHOLD)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

# Modularity Q (left axis)
l1, = ax1.plot(thr_vals, q_vals, marker="o", color="#D7191C",
               linewidth=2.2, markersize=7, label="Modularity Q")
ax1.scatter([PAPER_THRESHOLD], [q_vals[paper_idx]],
            s=130, color="#D7191C", edgecolors="black", linewidths=1.5, zorder=5)
ax1.annotate(f"Q={q_vals[paper_idx]:.3f}",
             xy=(PAPER_THRESHOLD, q_vals[paper_idx]),
             xytext=(6, -14), textcoords="offset points", fontsize=9,
             color="#D7191C",
             bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#ccc", alpha=0.85))

# Node Retention (right axis)
l2, = ax2.plot(thr_vals, ret_vals, marker="s", color="#2C7BB6",
               linewidth=2.2, markersize=7, linestyle="--", label="Node Retention")
ax2.scatter([PAPER_THRESHOLD], [ret_vals[paper_idx]],
            s=130, color="#2C7BB6", edgecolors="black", linewidths=1.5, zorder=5)
ax2.annotate(f"{ret_vals[paper_idx]:.1f}%",
             xy=(PAPER_THRESHOLD, ret_vals[paper_idx]),
             xytext=(6, 6), textcoords="offset points", fontsize=9,
             color="#2C7BB6",
             bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#ccc", alpha=0.85))

# Selected threshold line
ax1.axvline(PAPER_THRESHOLD, color="#555", linestyle=":", linewidth=1.5,
            label=f"Selected threshold ({PAPER_THRESHOLD})")

ax1.set_xlabel("Cosine Similarity Threshold", fontsize=11)
ax1.set_ylabel("Modularity Q", fontsize=11, color="#D7191C")
ax2.set_ylabel("Node Retention (%)", fontsize=11, color="#2C7BB6")
ax1.tick_params(axis="y", labelcolor="#D7191C")
ax2.tick_params(axis="y", labelcolor="#2C7BB6")
ax1.set_xticks(THRESHOLDS)
ax1.grid(axis="y", linestyle="--", alpha=0.35)

lines = [l1, l2,
         plt.Line2D([0], [0], color="#555", linestyle=":", linewidth=1.5,
                    label=f"Selected threshold ({PAPER_THRESHOLD})")]
ax1.legend(handles=lines, fontsize=9, loc="center left")
plt.title("Threshold Sensitivity: Modularity Q vs Node Retention",
          fontsize=12, fontweight="bold")
plt.tight_layout()
fig_path = f"{SAVE_DIR}/threshold_sensitivity_dualaxis.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"✅ Figure saved: {fig_path}")
plt.show()

# ── Verdict ───────────────────────────────────────────────────────────────────
row = summary_df[summary_df["threshold"] == PAPER_THRESHOLD].iloc[0]
print("\n── Threshold Selection Summary ──")
print(f"Selected threshold : {PAPER_THRESHOLD}")
print(f"Node retention     : {row['node_retention']:.1%}  (n = {row['n_nodes']:,})")
print(f"Modularity Q       : {row['modularity']:.4f}")
print(f"Communities        : {row['n_communities']}")
print(f"Top-5 share        : {row['top5_share']:.1%}")
print("\nData-driven summary for this refined cohort:")
print(
    f"At threshold={PAPER_THRESHOLD:.2f}, node retention="
    f"{row['node_retention']:.1%}, modularity Q={row['modularity']:.3f}, "
    f"communities={int(row['n_communities'])}, top-5 share={row['top5_share']:.1%}."
)
print(
    "Interpret the threshold sensitivity using the newly generated table/figure; "
    "do not reuse the old cohort's hard-coded retention or modularity values."
)
